# Recherche de parties d'échecs remarquables dans la base Lichess

<a id="sommaire"></a>
# Sommaire

**[0. Configuration](#0)**

**[I. Introduction](#I)**
- [A. Contexte](#I-A)
- [B. Objectif du projet](#I-B)
- [C. Architecture](#I-C)
- [D. Résultats](#I-D)

**[II. Méthode et vue d'ensemble qualitative du projet](#II)**
- [A. Le problème de l'aiguille dans une botte de foin](#II-A)
- [B. Présentation du dataset](#II-B)
- [C. « L'esthétisme » : comment quantifier un concept humain ?](#II-C)
- [D. Usage de l'IA générative](#II-D)

**[III. Présentation détaillée des trois couches](#III)**
- [A. Étape 1, Nettoyage et premières hypothèses](#III-A)
- [B. Étape 2, Second filtrage et évaluation des parties](#III-B)
- [C. Étape 3, PU learning, des quelques milliers de candidates à 20 parties](#III-C)

**[IV. Conclusion](#IV)**

<a id="0"></a>
# 0. Configuration

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, Image, Markdown, display

%matplotlib inline
pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_colwidth", 80)

RACINE = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent     # racine du dépôt
SRC = RACINE / "src"

MODE = "complet"            # "complet" : vos données ; "exemple" : jeu de données synthétique fourni (test rapide)

# --- Mode complet : à adapter
DONNEES = Path(os.environ.get("LICHESS_DATA", RACINE / "donnees"))     # dossier des données (hors dépôt git)
DUMP = "lichess_db_standard_rated_2026-01.pgn.zst"                    # dump mensuel, placé dans DONNEES
STOCKFISH = os.environ.get("STOCKFISH_PATH", "")                       # chemin de l'exécutable Stockfish
P = dict(tranche=500_000, bloc_lecture=100_000, n_min=5_000, m_min=50, p_seuil=1e-3,
         n_temoin=5_000, profondeur=12, seuil_pu=0.5, n_final=20, graine=0)

# --- Étapes à exécuter (False : relire et afficher les fichiers déjà produits)
LANCER = dict(filtrage=True, features=True, selection=True, scraping=True, temoin=True,
              augmentation=True, pu=True, echantillon=True, comparaison=True)

if MODE == "exemple":
    DONNEES = RACINE / "exemples" / "travail"
    DUMP = str(RACINE / "exemples" / "mini_dump.pgn.zst")
    P.update(tranche=100, bloc_lecture=200, n_min=50, m_min=20, p_seuil=0.1, n_temoin=200, profondeur=8)

DONNEES.mkdir(parents=True, exist_ok=True)
RES, SEL, PU = DONNEES / "resultats", DONNEES / "resultats" / "selection", DONNEES / "pu"
print(f"Mode : {MODE}\nDonnées : {DONNEES}\nDump : {DUMP}\nStockfish : {STOCKFISH or '(non renseigné)'}")

In [ ]:
ENV = {**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUTF8": "1", "MPLBACKEND": "Agg"}
if STOCKFISH:
    ENV["STOCKFISH_PATH"] = STOCKFISH


def lancer(script, *args):
    """Exécute src/{script} dans un processus séparé (comme dans un terminal) et affiche sa sortie en direct."""
    cmd = [sys.executable, "-u", str(SRC / script), *[str(a) for a in args]]
    print("$ python src/" + script + " " + " ".join(str(a) for a in args) + "\n")
    t = time.perf_counter()
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", env=ENV, cwd=RACINE) as p:
        for ligne in p.stdout:
            print(ligne, end="")
    duree = time.perf_counter() - t
    if p.returncode:
        raise RuntimeError(f"{script} s'est arrêté avec une erreur (code {p.returncode}) : voir la sortie ci-dessus.")
    print(f"\n[{script} terminé en {duree / 60:.1f} min]")


def liens(df, **kw):
    """Tableau HTML où les adresses de parties sont cliquables."""
    display(HTML(df.to_html(render_links=True, escape=True, index=False, **kw)))


def images(*motifs, largeur=None):
    for motif in motifs:
        for f in sorted(Path(RES).glob(motif)) if not Path(motif).is_absolute() else [Path(motif)]:
            if f.exists():
                display(Markdown(f"`{f.name}`"))
                display(Image(filename=str(f), width=largeur))

<a id="I"></a>
# I. Introduction

<a id="I-A"></a>
## A. Contexte

Chaque mois près de 100 millions de parties d'échecs sont jouées sur lichess.org, le deuxième site le plus populaire en la matière. Chaque mois les joueurs proposent et élisent au cours d'un vote la 'game of the month', à savoir la plus belle partie selon des critères humains subjectifs d'esthétisme : game of the month d'août 2026, https://lichess.org/@/Lichess/blog/lichess-game-of-the-month-august-26/ekVdAfDN.

<a id="I-B"></a>
## B. Objectif du projet

L'objectif du projet est de concevoir un algorithme prenant en entrée les 100 millions de parties mensuelles et renvoyant 20 parties esthétiques pour l'oeil humain qui pourraient être désignées 'game of the month'.

Au cours de ce travail nous concevrons cet algorithme et évaluerons ses performances.

<a id="I-C"></a>
## C. Architecture

Nous avons à faire à un problème de recherche d'aiguille dans une botte de foin. Ainsi, notre algorithme fonctionne comme un entonnoir : trois filtres successifs permettent de passer du dataset complet à l'échantillon final de 20 parties.

Plus le filtre traite un nombre important de parties plus il doit être léger en s'appuyant sur des opérations simples, et inversement plus il est proche de la sortie de l'entonnoir plus il peut utiliser des méthodes couteuses pour classifier les dernières parties.

La première couche  (partie III A) correspond au nettoyage des données et à la sélection des parties sur lesquelles nous souhaitons nous attarder permettant de passer de 100 à 3 millions de parties, la seconde (partie III B) utilise des propriétés statistiques de manière à passer de 3 millions à 3 000 parties, et la dernière utilise un modèle de Machine Learning pour obtenir 20 parties.

<a id="I-D"></a>
## D. Résultats

<a id="II"></a>
# II. Méthode et vue d'ensemble qualitative du projet

<a id="II-A"></a>
## A. Le problème de l'aiguille dans une botte de foin

Notre sujet est assez atypique, en effet en plus de réaliser des statistiques descriptives, le coeur de notre problématique est de concevoir un algorithme. Pour que le projet soit sérieux il convient de pouvoir évaluer ses performances. Définissons une métrique qui permet cela.

Cet algorithme tente de trouver des éléments rares (aiguilles) noyés dans un volume considérable d'information (la botte de foin). Ainsi nous utiliserons le taux de vrais positifs : le nombre de parties effectivement remarquables parmi les 20 renvoyées par l'algorithme.

Nous comparerons ce résultat au témoin servant de référence. Le témoin est obtenu en tirant au hasard 10 échantillons de 20 parties parmi les 100 millions de parties. Après avoir étudié la moyenne empirique et la distribution du témoin nous pourrons conclure que notre algorithme est meilleur (ou moins bon...) que le baseline random avec un certain intervalle de confiance.

Nota Bene : Faute de mieux, la classification binaire entre partie 'esthétique' et lambda a été faite à la main par un panel de 3 joueurs (classés 1200, 1500 et 2000 elo sur chess.com) à qui les 220 parties ont été montrées mélangées, une partie était 'esthétique' si au moins 2 des 3 joueurs humains la jugeaient telle quelle et lambda sinon.


<a id="II-B"></a>
## B. Présentation du dataset

**Source.** La base ouverte de Lichess (https://database.lichess.org, licence CC0) publie chaque mois l'intégralité des parties classées. Nous travaillons sur janvier 2026, soit environ 92 millions de parties.

**Format.** Chaque partie est enregistrée au format PGN, le standard des échecs :
- des **en-têtes** : joueurs et classement Elo, résultat, cadence, ouverture (code ECO), lien vers la partie ;
- la **suite des coups**, accompagnée pour une minorité de parties de l'évaluation du moteur après chaque coup (lorsqu'un joueur a demandé l'analyse).

**Volume.** Le fichier compressé pèse plusieurs dizaines de Go : il reste sur notre machine et il est lu en flux, partie par partie, sans jamais être chargé en entier. Les résultats intermédiaires sont enregistrés au format Parquet, compact et rapide à relire.

**Données complémentaires.** Les « parties du mois » publiées sur le blog de Lichess sont les seuls exemples de parties jugées belles par des humains : elles servent de référence à l'étape 3.

<a id="II-C"></a>
## C. « L'esthétisme » : comment quantifier un concept humain ?

Une belle partie ne se mesure pas directement. Nous l'approchons par des **indicateurs observables**, inspirés de ce que valorisent joueurs et commentateurs :

- **les sacrifices** : le vainqueur accepte de perdre du matériel pour prendre l'initiative ;
- **les retournements** : l'avantage change de camp, l'issue reste longtemps incertaine ;
- **l'intensité tactique** : échecs, captures en série, mats spectaculaires, promotions inhabituelles ;
- **la précision sous pression** : le vainqueur trouve les coups justes dans une position difficile.

Deux idées structurent la démarche.

**Rare n'est pas beau, mais le beau est rare.** On isole d'abord les parties statistiquement inhabituelles (étape 2), puis on apprend, à partir d'exemples humains, à séparer l'inhabituel « beau » de l'inhabituel « bizarre » (étape 3).

**L'inhabituel est relatif.** Une partie n'est comparée qu'aux parties de la même ouverture, et le niveau des joueurs n'entre pas dans le score : on cherche une partie remarquable pour ce qu'elle contient, pas parce qu'elle oppose des joueurs forts.

**Limite.** Ces indicateurs voient mal la beauté positionnelle (manœuvre lente, coup calme décisif), qui ne laisse ni trace matérielle ni bascule nette de l'évaluation.

<a id="II-D"></a>
## D. Usage de l'IA générative

Il convient enfin de détailler l’usage qui a été fait de l’IA générative pour réaliser ce projet. Avant tout, l’IA nous a permis de trouver rapidement des datasets adaptés à notre projet sur internet. Concernant le projet en lui-même, nous avons codé une première version permettant d’avoir un squelette déjà opérationnel sur un nombre de parties restreint. Cette première version manuelle était d’ailleurs nécessaire pour garder la main sur l’ensemble des critères choisis, qui visent in fine à détecter des “belles parties”, à l'aune de notre propre appréciation esthétique. Ce n’est que par la suite que nous avons employé Claude notamment, pour débugger certains blocs, et scaler notre projet afin de le rendre applicable à des millions de parties (ex : multiprocessing dans analyse_features.py). La rédaction du README a enfin été essentiellement manuelle pour s’assurer de la clarté et de la pertinence des explications au regard de nos propres impressions. 

<a id="III"></a>
# III. Présentation détaillée des trois couches

| Étape | Script | Résultat |
|---|---|---|
| 1. Filtrage du dump | `filtrer_parties.py` | parties retenues, avec leurs évaluations |
| 1. Features | `analyse_features.py` | une soixantaine de features par partie, statistiques et graphiques |
| 2. Sélection | `selection_pot.py` | quelques milliers de parties atypiques (Mahalanobis + POT, par code ECO) |
| 3a. Parties du mois | `scraping.py` | parties du mois Lichess (positifs) |
| 3b. Témoin | `temoin.py` | parties tirées au hasard (non étiquetées) |
| 3c. Augmentation | `augmentation.py` | mêmes évaluations Stockfish et mêmes features pour les trois groupes |
| 3d. PU learning | `pu_learning.py` | candidates étiquetées positives |
| 3e. Échantillon final | `echantillon_final.py` | 20 parties |
| Comparaison | `comparaison.py` | 200 parties quelconques (10 blocs de 20) |

<a id="III-A"></a>
## A. Étape 1, Nettoyage et premières hypothèses

**Objectif.** Réduire fortement le volume avec des tests très peu coûteux (on ne lit que les en-têtes et la ligne des coups), sans risquer d'écarter une partie remarquable.

**Parties conservées :**
- **une issue nette** : victoire par mat ou par abandon ; les nulles et les pertes au temps sont écartées, car plusieurs indicateurs reposent sur le vainqueur ;
- **un niveau minimal et un duel équilibré** : deux joueurs au-dessus de 1500 Elo, avec moins de 400 points d'écart ;
- **une vraie partie** : entre 20 et 120 coups, en cadence définie (hors parties par correspondance) ;
- **une évaluation disponible** : indispensable aux indicateurs de retournement et de précision.

**Un biais assumé.** Seules les parties qu'un joueur a fait analyser possèdent une évaluation. C'est plutôt un bon signal (on analyse surtout une partie qu'on juge intéressante), mais la population étudiée n'est plus représentative de l'ensemble du site : les conclusions des étapes suivantes portent sur elle.

In [ ]:
if LANCER["filtrage"]:
    lancer("filtrer_parties.py", "--dossier", DONNEES, "--dump", DUMP, "--tranche", P["tranche"])

import pyarrow.parquet as pq
n_filtrees = pq.ParquetFile(DONNEES / "parties_filtrees.parquet").metadata.num_rows
print(f"Parties retenues : {n_filtrees:,}")
comptes = pd.read_csv(DONNEES / "comptes_par_tranche.csv")
if len(comptes) > 1:
    display(comptes["n_gardees"].describe().to_frame("parties gardées par tranche").T.round(1))
    ax = comptes.plot(x="tranche", y="n_gardees", figsize=(10, 3), legend=False, title="Parties gardées par tranche")
    ax.set_ylabel("parties gardées"); plt.show()

**Résultat.** Il reste *3 124 123* parties, soit *3,3 %* du dump.

Le dump est lu par tranches de 500 000 parties, et le nombre de parties conservées varie très peu d'une tranche à l'autre (coefficient de variation d'environ 3 %).

Le volume en sortie est donc prévisible, ce qui nous a permis de dimensionner l'étape suivante pour environ 3 millions de parties.

<a id="III-B"></a>
## B. Étape 2, Second filtrage et évaluation des parties

**Objectif.** Passer d'environ 3 millions à quelques milliers de parties, sans moteur d'échecs, en repérant celles qui s'écartent nettement de la norme de leur ouverture.

**Décrire chaque partie.** Une soixantaine d'indicateurs sont calculés pour chaque partie (`features_core.py`), en une seule lecture des coups :
- **matériel et tactique** : captures, échecs, sacrifices apparents, remontées, roques, pions passés ;
- **évaluation** : précision des deux camps, gaffes, retournements d'avantage, sacrifices confirmés par le moteur.

Chaque partie devient un point dans un espace d'une soixantaine de dimensions. L'Elo, la longueur et la fréquence de l'ouverture décrivent le contexte : ils sont calculés, mais n'entrent pas dans le score.

*NB : les statistiques descriptives qui ont guidé nos choix ont d'abord été établies sur un échantillon de 200 000 parties (environ 6 000 après filtrage), avant le passage à l'échelle.*

In [ ]:
if LANCER["features"]:
    lancer("analyse_features.py", "--dossier", DONNEES, "--bloc-lecture", P["bloc_lecture"])

resume = pd.read_csv(RES / "resume_statistiques.csv", index_col=0)
display(resume.round(3))

**Lecture du tableau.** Trois traits se dégagent :
- les indicateurs d'événements rares (sacrifices, remontées, sous-promotions) valent zéro pour la grande majorité des parties ;
- les distributions sont très asymétriques : quelques parties concentrent les valeurs extrêmes ;
- les compteurs (captures, échecs) augmentent mécaniquement avec la longueur de la partie.

In [ ]:
images("distributions_*.png")

**Des distributions à queue lourde.** L'immense majorité des parties est concentrée près de zéro, tandis qu'une poignée atteint des valeurs extrêmes : ce sont précisément celles qui nous intéressent.

Deux conséquences :
- une transformation logarithmique atténue le poids des valeurs extrêmes avant le calcul des distances ;
- la modélisation de ces queues relève de la théorie des valeurs extrêmes (EVT), et non d'une loi normale ajustée sur le centre de la distribution.

In [ ]:
images("matrice_correlation.png", "nuages_de_points_materiel.png", "nuages_de_points_evaluation.png")

**Des indicateurs fortement corrélés.** Plusieurs familles mesurent des phénomènes voisins : la longueur et les compteurs de coups, les différentes mesures de précision, les sacrifices matériels et les sacrifices confirmés par le moteur.

Une distance euclidienne compterait plusieurs fois le même phénomène et mélangerait des échelles très différentes (indicateurs binaires, compteurs, pourcentages).

La **distance de Mahalanobis** corrige ces deux défauts : elle remet chaque variable à la même échelle et tient compte des corrélations, si bien que des indicateurs redondants ne pèsent pas plus qu'un seul.

### Sélection des parties atypiques (`selection_pot.py`)

**Une norme par ouverture.**

![Comparaison entre ouvertures](images/graph3_comparaison_eco.png)

Les distributions diffèrent nettement d'une ouverture à l'autre : une ouverture tranchante produit naturellement plus de captures et d'échecs. Chaque partie est donc comparée aux parties de **la même ouverture** (code ECO). Seules les ouvertures d'au moins 5 000 parties sont conservées : en dessous, la norme est mal estimée.

**Un score d'atypicité robuste.** La distance de Mahalanobis mesure l'éloignement de chaque partie au centre de son ouverture. Pour que les parties extrêmes ne déplacent pas elles-mêmes ce centre, il est estimé en écartant à plusieurs reprises les 10 % de parties les plus éloignées.

**Un seuil justifié par l'EVT.** Plutôt qu'un « top 100 » arbitraire, on modélise la queue de la distribution des distances (méthode POT : loi de Pareto généralisée ajustée au-delà du 95e percentile). Chaque partie reçoit une probabilité de rareté : la chance qu'une partie de la même ouverture soit au moins aussi atypique. Les parties sous le seuil de 1 pour 1 000 sont retenues.

L'intérêt de l'EVT : estimer de façon fiable ce qui se passe tout au bout de la queue, là où il ne reste presque plus de données, avec un seuil comparable entre petites et grandes ouvertures.

In [ ]:
if LANCER["selection"]:
    lancer("selection_pot.py", "--dossier", DONNEES, "--n-min", P["n_min"], "--m-min", P["m_min"],
           "--p-seuil", P["p_seuil"])

resume_eco = pd.read_csv(SEL / "resume_par_groupe.csv")
base = pd.read_parquet(SEL / "base_selection.parquet")
display(resume_eco["statut"].value_counts().to_frame("codes ECO"))
print(f"Parties retenues (base unique) : {len(base):,}")
display(resume_eco[resume_eco.statut == "ok"].sort_values("n_retenues", ascending=False).head(20).round(3))

**Lecture.** Par construction, le seuil retient environ une partie sur mille dans chaque ouverture : c'est lui qui fixe le volume final (quelques milliers de parties), l'EVT garantissant que ce sont bien les plus atypiques.

Le résumé par ouverture sert de contrôle :
- un nombre de parties retenues très éloigné du nombre attendu signale une queue mal modélisée ;
- l'indicateur `ks` repère les ouvertures où la loi de Pareto colle mal aux données.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
base["ECO"].value_counts().head(30).plot.bar(ax=axes[0], title="Parties retenues par code ECO (30 premiers)")
base["rarete"].plot.hist(bins=40, ax=axes[1], title="Rareté des parties retenues (-log10 p)")
plt.tight_layout(); plt.show()
liens(base.sort_values("rarete", ascending=False).head(15)[
    ["Site", "ECO", "Result", "rarete", "rang_groupe", "features_extremes"]].round(2))

**Modélisation de la queue.**

![Ajustement EVT/POT](images/graph4_evt_pot.png)

Au-delà du seuil, la loi de Pareto suit bien la queue des distances : les probabilités de rareté restent fiables jusqu'aux parties les plus extrêmes.

**Comprendre une sélection.** Pour chaque partie retenue, la colonne `features_extremes` indique les trois indicateurs qui l'éloignent le plus de la norme de son ouverture : c'est l'explication lisible du choix.

**Limite.** La distance ignore le sens de l'écart : une partie peut être atypique par excès (sacrifices en série) comme par défaut (aucune capture, finale interminable). Cette étape sélectionne donc l'**inhabituel**, pas encore le beau : c'est le rôle de l'étape 3.

<a id="III-C"></a>
## C. Étape 3, PU learning, des quelques milliers de candidates à 20 parties

**Le problème.** L'étape 2 renvoie des parties inhabituelles, pas forcément belles. Pour trier, il faut apprendre ce qu'un humain trouve beau. Or nous n'avons que des exemples **positifs**, les parties du mois, et aucun exemple négatif certain : une partie quelconque peut être belle sans avoir été primée.

C'est le cadre du **PU learning** (*Positive–Unlabeled*), qui apprend à partir de deux groupes :
- **les positifs** : les parties du mois de Lichess ;
- **les non étiquetés** : un échantillon de parties ordinaires, qui contient peut-être quelques belles parties passées inaperçues.

Le modèle obtenu est ensuite appliqué aux candidates de l'étape 2.

### 3a. Parties du mois (scraping du blog Lichess)

Le script parcourt le blog de Lichess et récupère, pour chaque mois, la partie élue et les éventuelles finalistes.

Janvier et février 2026 sont exclus : la partie de janvier appartient au mois étudié et ne doit pas servir à l'apprentissage. Elle peut au contraire servir de test : l'algorithme la retrouve-t-il ?

**Limite.** Environ une partie par mois, soit quelques dizaines d'exemples : le modèle devra rester simple.

In [ ]:
PU.mkdir(parents=True, exist_ok=True)
if LANCER["scraping"]:
    if MODE == "exemple":                          # pas de réseau en mode exemple : parties du mois synthétiques
        shutil.copy(RACINE / "exemples" / "gotm_exemple.pgn", PU / "gotm.pgn")
        print("Mode exemple : exemples/gotm_exemple.pgn copié dans", PU / "gotm.pgn")
    else:
        lancer("scraping.py", "--dossier", DONNEES)

if (PU / "gotm_index.csv").exists() and MODE != "exemple":
    index = pd.read_csv(PU / "gotm_index.csv")
    print(f"Parties du mois récupérées : {len(index)} ({index['mois'].nunique()} mois)")
    liens(index[["mois", "Site", "White", "Black", "Result", "n_plies"]])
else:
    print("Parties du mois :", open(PU / "gotm.pgn", encoding="utf-8").read().count("[Event "))

### 3b. Groupe témoin

Le témoin représente la population dans laquelle on cherche : 5 000 parties tirées au hasard parmi celles de l'étape 1, candidates exclues.

C'est environ cent fois plus que de parties du mois : assez pour décrire une partie « ordinaire », sans trop alourdir l'analyse Stockfish de l'étape suivante.

In [ ]:
if LANCER["temoin"]:
    lancer("temoin.py", "--dossier", DONNEES, "--n", P["n_temoin"], "--graine", P["graine"])
temoin = pd.read_parquet(PU / "temoin.parquet")
print(f"Parties témoin : {len(temoin):,}")
display(temoin.drop(columns=["Moves_clean"], errors="ignore").head())

### 3c. Augmentation : évaluation Stockfish et recalcul des indicateurs

Les trois groupes n'ont pas des évaluations comparables : les parties du mois n'en ont souvent pas, et celles de Lichess proviennent d'analyses de profondeurs différentes.

Toutes les parties sont donc réanalysées par **le même moteur, à la même profondeur**, puis leurs indicateurs recalculés par le même code. Sans cette précaution, le modèle pourrait distinguer les groupes par une simple différence de mesure plutôt que par le contenu des parties.

In [ ]:
if LANCER["augmentation"]:
    args = ["--dossier", DONNEES, "--profondeur", P["profondeur"]]
    lancer("augmentation.py", *args, *(["--stockfish", STOCKFISH] if STOCKFISH else []))
aug = pd.read_parquet(PU / "features_augmentees.parquet")
display(aug.groupby("source").agg(parties=("cle", "size"), evaluees=("acpl_gagnant", "count")))
comparer = ["acpl_gagnant", "accuracy_gagnant", "n_gaffes_perdant", "win_min_gagnant", "n_sacrifices_pieces",
            "n_plies_sacrifice_valide", "n_brillants_gagnant", "swing_max", "n_echecs", "n_plies"]
display(aug.groupby("source")[comparer].median().T.round(2).rename_axis("médiane par groupe"))

**Lecture.** Le tableau des médianes montre en quoi une partie du mois diffère d'une partie ordinaire. *(à compléter : par exemple plus d'échecs, moins de gaffes du vainqueur, davantage de sacrifices confirmés)*

Ce sont ces écarts que le modèle va apprendre.

### 3d. PU learning

**Principe** (méthode d'Elkan et Noto, 2008). Un classifieur apprend d'abord à distinguer les parties du mois des parties témoins. Comme le témoin contient sans doute quelques belles parties non repérées, ses probabilités sont sous-estimées : on les corrige par un facteur estimé sur des parties du mois que le modèle n'a pas vues. Une candidate est étiquetée positive si sa probabilité corrigée dépasse 0,5.

**Choix du modèle.** Avec quelques dizaines d'exemples positifs, un modèle simple (régression logistique régularisée) généralise mieux qu'un modèle complexe.

L'Elo, la longueur et l'ouverture sont exclus : les parties du mois opposent des joueurs bien plus forts que la moyenne, et le modèle apprendrait sinon le niveau plutôt que la beauté.

In [ ]:
if LANCER["pu"]:
    lancer("pu_learning.py", "--dossier", DONNEES, "--seuil", P["seuil_pu"], "--graine", P["graine"])
r = json.load(open(PU / "pu_resume.json", encoding="utf-8"))
display(Markdown(f"### Nombre de candidates étiquetées positives : **{r['n_candidates_positives']}** "
                 f"sur {r['n_candidates']:,}"))
display(pd.Series(r).to_frame("valeur"))
display(Image(filename=str(PU / "pu_scores.png")))
if (PU / "pu_coefficients.csv").exists():
    display(pd.read_csv(PU / "pu_coefficients.csv").head(15).round(3))

**Lecture.**
- Le **pouvoir de séparation** (AUC hors échantillon) indique si les parties du mois se distinguent bien des parties ordinaires ; il ne garantit pas que cette distinction repose sur l'esthétique.
- La **part du témoin étiquetée positive** donne une idée du taux de fausses alertes sur des parties ordinaires.
- Les **coefficients** montrent quels indicateurs le modèle associe à une belle partie : on peut les confronter aux critères de la partie II.C.

*(à compléter : résultats et interprétation)*

**Limite.** La méthode suppose que les parties du mois sont un échantillon représentatif des belles parties. Ce n'est qu'approximatif : elles sont élues par vote, parmi des parties déjà visibles. Avec si peu d'exemples, la probabilité obtenue sert surtout à **classer** les candidates.

### 3e. Échantillon final (20 parties)

S'il y a plus de 20 candidates positives, on compose un échantillon mixte :
- **10 parties parmi les plus atypiques** de l'étape 2 ;
- **10 parties plus communes**, réparties sur la moitié basse des raretés ;
- au plus 3 parties par ouverture.

Ce mélange évite de ne montrer que des cas extrêmes, souvent atypiques pour de mauvaises raisons, et permet de comparer les deux profils lors de l'évaluation. S'il y en a 20 ou moins, toutes sont conservées.

In [ ]:
if LANCER["echantillon"]:
    lancer("echantillon_final.py", "--dossier", DONNEES, "--n", P["n_final"])
final = pd.read_csv(PU / "echantillon_final.csv")
liens(final[[c for c in ["rang", "profil", "Site", "ECO", "Result", "rarete", "p_pu", "features_extremes"]
             if c in final]].round(3))

**Avis sur les 20 parties :**

*(vos observations ici)*

### Évaluation : comparaison avec 200 parties tirées au hasard

**Protocole.** 200 parties sont tirées au hasard parmi **toutes** les parties du mois, sans aucun filtre, en 10 blocs de 20. Les 220 parties (les 20 sélectionnées et les 200 tirées au hasard) sont présentées mélangées, sans indication de provenance, au panel de trois joueurs ; une partie est jugée esthétique si au moins deux d'entre eux la retiennent.

**Comparaison.** On compare la part de parties esthétiques dans l'échantillon final à celle des parties tirées au hasard ; l'écart entre les 10 blocs montre ce que produit le hasard sur 20 parties. Un test exact de Fisher indique si l'écart observé dépasse ce que le hasard peut expliquer.

**À garder en tête.** Sur 20 parties, seul un écart net est détectable : si une partie sur vingt est jugée belle au hasard, il en faut au moins quatre dans l'échantillon final pour conclure.

In [ ]:
if LANCER["comparaison"]:
    lancer("comparaison.py", "--dossier", DONNEES, "--dump", DUMP, "--graine", P["graine"])
comp = pd.read_csv(DONNEES / "comparaison" / "comparaison_200.csv")
tableau = comp.pivot(index="position", columns="bloc", values="Site")
tableau.columns = [f"bloc {b}" for b in tableau.columns]
display(HTML(tableau.to_html(render_links=True)))

**Comparaison avec l'échantillon final :**

*(vos observations ici)*

<a id="IV"></a>
# IV. Conclusion

*(vos conclusions ici)*